# 03 — Dimensões e Fato (exercício)

**Objetivo:** praticar star schema e chaves substitutas determinísticas.

Execute os notebooks em ordem.

## Onde estamos no pipeline

![Star schema alvo](../docs/images/image.png)

Este é o estado final que vamos atingir ao terminar a sessão: uma fato `fact_report` com chaves estrangeiras para quatro dimensões. Cada notebook contribui com uma camada do caminho até esse esquema.

Posição deste notebook (camada **silver / modelo dimensional**):

> Fonte → Bronze → Staging → **Dim/Fact** → Checks → Profiling → Marts/Views → Dashboard

Este é o notebook em que o esquema acima é literalmente construído: `dim_reporter`, `dim_team`, `dim_weakness`, `dim_structured_scope` e `fact_report`. As surrogate keys vêm de `hash(...)` sobre o JSON original — assim a fato sempre aponta para a mesma linha de dimensão quando o conteúdo é igual.

In [ ]:
# Parametros usados pelo Papermill/Airflow.
run_date = None
project_root = None

from pathlib import Path
import sys

for candidato in [Path.cwd(), *Path.cwd().parents]:
    caminhos_common = [
        candidato / "_common.py",
        candidato / "aula02" / "_common.py",
        candidato / "exercicios" / "_common.py",
        candidato / "sessao-02-data-architecture" / "_common.py",
        candidato / "sessao-02-data-architecture" / "exercicios" / "_common.py",
    ]
    encontrado = next((p for p in caminhos_common if p.exists()), None)
    if encontrado:
        sys.path.insert(0, str(encontrado.parent))
        break
else:
    raise FileNotFoundError("Nao encontrei _common.py a partir do diretorio atual.")

from _common import configurar_paths, conectar_duckdb
paths = configurar_paths(project_root)
globals().update(paths)

print(f"Raiz do projeto: {PROJECT_ROOT}")
print(f"Banco DuckDB: {DB_PATH}")


## Exercício
Complete as chaves substitutas com `hash(...)` e monte a tabela fato.

In [ ]:
con = conectar_duckdb(DB_PATH)
print("Conexão DuckDB aberta.")

In [ ]:
con.sql("""
CREATE OR REPLACE TABLE dim_reporter AS
SELECT DISTINCT
    hash(reporter_json) AS reporter_id,
    username,
    verified
FROM stg_reporter;
""")

con.sql("""
CREATE OR REPLACE TABLE dim_team AS
SELECT DISTINCT
    hash(team_json) AS team_id,
    id,
    handle,
    offers_bounties
FROM stg_team;
""")

con.sql("""
CREATE OR REPLACE TABLE dim_weakness AS
SELECT DISTINCT
    hash(weakness_json) AS weakness_id,
    id,
    name
FROM stg_weakness;
""")

con.sql("""
CREATE OR REPLACE TABLE dim_structured_scope AS
SELECT DISTINCT
    hash(structured_scope_json) AS asset_id,
    asset_identifier,
    asset_type,
    max_severity
FROM stg_asset;
""")

con.sql("""
CREATE OR REPLACE TABLE fact_report AS
SELECT
    r.report_id,
    r.title,
    r.substate,
    r.visibility,
    r.has_bounty,
    r.vote_count,
    r.created_at,
    r.disclosed_at,
    r.vulnerability_information,
    hash(r.reporter_json) AS reporter_id,
    hash(r.team_json) AS team_id,
    hash(r.weakness_json) AS weakness_id,
    hash(r.structured_scope_json) AS asset_id
FROM stg_reports r;
""")

for tabela in ["dim_reporter", "dim_team", "dim_weakness", "dim_structured_scope", "fact_report"]:
    total = con.sql(f"SELECT COUNT(*) FROM {tabela}").fetchone()[0]
    print(f"{tabela}: {total:,} linhas")
